# UK Used Car Price Prediction: 03. Preprocessing dan Feature Engineering

Notebook ini membagi data bersih menjadi train/test, membuat pipeline preprocessing yang dapat diulang, dan melatih baseline. Semua preprocessing di-fit hanya pada data training untuk mencegah data leakage.

**Input:** `data/processed/cars_clean.csv` dari notebook 01.  \n
**Output:** hasil baseline dan `models/03_ridge_baseline.joblib`.

Notebook berikutnya (`04_modeling_evaluation.ipynb`) memakai split dan metrik yang sama untuk membandingkan model lanjutan.

## 1. Setup dan Load Data

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
DATA = Path("../data/processed/cars_clean.csv")
MODEL_PATH = Path("../models/03_ridge_baseline.joblib")
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import add_car_age

df = pd.read_csv(DATA)
print(f"Data bersih: {df.shape[0]:,} baris x {df.shape[1]} kolom")
display(df.head())

Data bersih: 97,411 baris x 10 kolom


,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,brand
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4,audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,audi


## 2. Train/Test Split

Kolom `price` adalah target. Test set disisihkan sebelum preprocessing dan tidak dipakai saat memilih transformasi maupun parameter baseline.

In [2]:
TARGET = "price"
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame(
    {"baris": [len(X_train), len(X_test)],
     "proporsi": [len(X_train) / len(X), len(X_test) / len(X)],
     "median_price": [y_train.median(), y_test.median()]},
    index=["train", "test"],
)
display(split_summary.style.format({"proporsi": "{:.1%}", "median_price": "£{:,.0f}"}))

assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_test) == len(df)

,baris,proporsi,median_price
train,77928,80.0%,"£14,480"
test,19483,20.0%,"£14,400"


## 3. Pipeline Preprocessing

`year` diubah menjadi `car_age` agar maknanya lebih langsung untuk model, kemudian kolom `year` asli dihapus. Fitur numerik diskalakan untuk Ridge. Fitur kategorikal di-one-hot encode; kategori model dengan frekuensi kurang dari 10 digabung oleh `OneHotEncoder(min_frequency=10)`. Kategori yang belum pernah muncul saat training diabaikan saat prediksi, bukan menyebabkan error.

In [3]:
numeric_features = ["car_age", "mileage", "tax", "mpg", "engineSize"]
categorical_features = ["brand", "model", "transmission", "fuelType"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(
            handle_unknown="ignore", min_frequency=10
        ), categorical_features),
    ],
    remainder="drop",
)

feature_pipeline = Pipeline(
    steps=[
        ("add_car_age", FunctionTransformer(add_car_age, validate=False)),
        ("preprocess", preprocessor),
    ]
)

X_train_ready = feature_pipeline.fit_transform(X_train)
X_test_ready = feature_pipeline.transform(X_test)
print(f"Matriks training: {X_train_ready.shape}")
print(f"Matriks test    : {X_test_ready.shape}")

model_encoder = feature_pipeline.named_steps["preprocess"].named_transformers_["categorical"]
encoded_names = model_encoder.get_feature_names_out(categorical_features)
model_encoded_features = [name for name in encoded_names if name.startswith("model_")]
print(f"Kolom hasil encoding model (setelah min_frequency=10): {len(model_encoded_features)}")

Matriks training: (77928, 178)
Matriks test    : (19483, 178)
Kolom hasil encoding model (setelah min_frequency=10): 155


## 4. Baseline

Dua pembanding dipakai:

1. **Median dummy:** selalu memprediksi median harga training.
2. **Ridge + log target:** baseline linear yang memakai semua fitur hasil pipeline. `TransformedTargetRegressor` menerapkan `log1p(price)` saat fit dan mengembalikan prediksi ke satuan pound saat prediksi.

Nilai `alpha=10` adalah baseline tetap, bukan hasil tuning; tuning dilakukan pada tahap modelling.

In [4]:
dummy = DummyRegressor(strategy="median")
dummy.fit(X_train, y_train)

ridge_log = TransformedTargetRegressor(
    regressor=Pipeline(
        steps=[
            ("features", feature_pipeline),
            ("model", Ridge(alpha=10.0, solver="lsqr")),
        ]
    ),
    func=np.log1p,
    inverse_func=np.expm1,
)
ridge_log.fit(X_train, y_train)

predictions = {
    "Median dummy": dummy.predict(X_test),
    "Ridge + log(price)": ridge_log.predict(X_test),
}

def score_regression(y_true, y_pred):
    return {
        "MAE (£)": mean_absolute_error(y_true, y_pred),
        "RMSE (£)": root_mean_squared_error(y_true, y_pred),
        "R²": r2_score(y_true, y_pred),
    }

results = pd.DataFrame(
    {name: score_regression(y_test, pred) for name, pred in predictions.items()}
).T.sort_values("MAE (£)")
display(results.style.format({"MAE (£)": "£{:,.0f}", "RMSE (£)": "£{:,.0f}", "R²": "{:.3f}"}))

,MAE (£),RMSE (£),R²
Ridge + log(price),"£1,651","£3,090",0.905
Median dummy,"£6,854","£10,271",-0.051


## 5. Simpan Baseline dan Catatan untuk Tahap Berikutnya

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(ridge_log, MODEL_PATH)
print(f"Baseline tersimpan: {MODEL_PATH}")

best_baseline = results.index[0]
print(f"Baseline terbaik berdasarkan MAE test: {best_baseline}")

Baseline tersimpan: ..\models\03_ridge_baseline.joblib
Baseline terbaik berdasarkan MAE test: Ridge + log(price)

Tahap 04: bandingkan Ridge dengan model tree-based (misalnya Random Forest dan XGBoost),
lakukan tuning hanya melalui cross-validation pada training set, lalu evaluasi sekali pada test set ini.
